# EDA + Baseline — Previsão de Churn

Tech Challenge Fase 1 · Pós Tech ML Engineering (FIAP)

Este notebook cobre o entregável da Etapa 1: exploração de dados e baseline
com Regressão Logística. A lógica de limpeza usada aqui é a mesma do módulo
`churn_prediction.preprocessing`, reaproveitada no treino produtivo.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

In [ ]:
df = pd.read_csv("../data/raw/Telco-Customer-Churn.csv")
df.shape

In [ ]:
df.head(3)

In [ ]:
df.info()

## Achados de qualidade de dados

- `TotalCharges` vem como string, com 11 linhas vazias — todas com `tenure=0`
  (clientes recém-chegados, sem ciclo de cobrança fechado ainda).
- `customerID` é único (sem duplicatas) — não é feature preditiva.
- Nenhuma linha duplicada.

In [ ]:
coerced = pd.to_numeric(df["TotalCharges"], errors="coerce")
problematic = df[coerced.isna()]
print(f"Linhas problemáticas em TotalCharges: {len(problematic)}")
problematic[["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]]

In [ ]:
print(df["Churn"].value_counts())
print(df["Churn"].value_counts(normalize=True).round(3))

## Limpeza (mesma lógica de `churn_prediction.preprocessing.clean_raw_dataframe`)

In [ ]:
import sys
sys.path.insert(0, "../src")
from churn_prediction.preprocessing import clean_raw_dataframe, split_features_target

df_clean = clean_raw_dataframe(df)
df_clean["TotalCharges"].isna().sum()

In [ ]:
ax = df["Churn"].value_counts().plot(kind="bar", color=["#4C72B0", "#DD8452"])
ax.set_title("Distribuição da variável alvo (Churn)")
ax.set_ylabel("Número de clientes")
plt.show()

In [ ]:
categorical_cols = ["Contract", "InternetService", "PaymentMethod", "gender", "SeniorCitizen"]

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, col in zip(axes.flatten(), categorical_cols):
    pd.crosstab(df[col], df["Churn"], normalize="index").plot(kind="bar", stacked=True, ax=ax)
    ax.set_title(col)
    ax.legend(title="Churn", fontsize=8)
fig.delaxes(axes.flatten()[-1])
plt.tight_layout()
plt.show()

## Baseline — Regressão Logística

Usamos o mesmo pipeline de `churn_prediction.train`, reproduzido aqui para
documentar e visualizar o resultado dentro do notebook (a versão que roda em
CI/produção fica em `src/churn_prediction/train.py`).

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, recall_score, classification_report
from churn_prediction.preprocessing import build_full_pipeline

X, y = split_features_target(df_clean)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

pipeline = build_full_pipeline(
    LogisticRegression(random_state=RANDOM_STATE, max_iter=1000, class_weight="balanced")
)
pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]

print("ROC-AUC:", roc_auc_score(y_test, y_proba))
print("PR-AUC:", average_precision_score(y_test, y_proba))
print("F1:", f1_score(y_test, y_pred))
print("Recall (Churn):", recall_score(y_test, y_pred))
print()
print(classification_report(y_test, y_pred))

## Próximos passos (Etapa 2)

- Treinar Random Forest / ensemble e MLPClassifier com o mesmo split e seed.
- Comparar as 3 abordagens nas 4 métricas acima + análise de custo de negócio.
- Escolher o modelo campeão e salvá-lo em `models/`.